# HOI Annotator
Annotate Human-Object Interaction intervals per clip and write `hoi-anns.json` next to the clip's `anns.json`.

## Output of (`hoi-anns.json`)
```json
{
  "annotations": [
    {
      "id": 0,
      "person_track_id": 1,
      "object_track_id": 5,
      "action_id": 3,
      "start_frame": 12,
      "end_frame": 78
    }
  ]
}

## Imports

In [1]:
import sys
import json
from pathlib import Path

import cv2
import ipywidgets as widgets
from IPython.display import display, clear_output
from matplotlib import pyplot as plt

In [2]:
# Set execution root in the project root
sys.path.insert(0, str(Path.cwd().parent.parent.parent))

In [3]:
from src.utils.config.config import Config

## Config

In [4]:
config_loader = Config()
cfg = config_loader.load_config()

dataset_root = cfg.project_root / cfg.paths.dataset / "CafeV1"
clips_root   = dataset_root / "Clips"

# Load action classes
with open(dataset_root / "action_classes.json", "r") as f:
    ACTION_NAME_TO_ID = json.load(f)

ACTION_ID_TO_NAME = {v: k for k, v in ACTION_NAME_TO_ID.items()} # Reverse



In [5]:
# COCO category id -> Class name
OBJECT_CATEGORY_NAMES = {
    1: "person", 
    2: "laptop", 
    3: "cell phone", 
    4: "book"
 }

## Helpers

In [6]:
def list_viewpoints():
    return sorted([p.name for p in clips_root.iterdir() if p.is_dir()], key=int)

def list_clips(viewpoint):
    vp_path = clips_root / viewpoint
    return sorted([p.name for p in vp_path.iterdir() if p.is_dir()], key=int)

def clip_dir(viewpoint, clip):
    return clips_root / viewpoint / clip

def load_clip(viewpoint, clip):
    """Returns (frames_by_number, anns_by_frame_number, hoi_path)."""
    cdir = clip_dir(viewpoint, clip)
    images_dir = cdir / "images"
    anns_path = cdir / "anns.json"

    frames_by_number = {
        int(p.stem.split("_")[1]): p
        for p in images_dir.iterdir() if p.is_file()
    }

    with open(anns_path, "r") as f:
        coco = json.load(f)

    image_id_to_frame_no = {}
    for img in coco.get("images", []):
        stem = Path(img["file_name"]).stem
        image_id_to_frame_no[img["id"]] = int(stem.split("_")[1])

    anns_by_frame_number = {}
    for ann in coco.get("annotations", []):
        fn = image_id_to_frame_no.get(ann["image_id"])
        if fn is None:
            continue
        anns_by_frame_number.setdefault(fn, []).append(ann)

    hoi_path = cdir / "hoi-anns.json"
    return frames_by_number, anns_by_frame_number, hoi_path

def get_track_id(ann):
    # CVAT exports inconsistently: sometimes int, sometimes str ("" for untracked)
    tid = ann.get("attributes", {}).get("track_id")
    if tid is None:
        tid = ann.get("track_id")
    if tid is None or tid == "":
        return None
    try:
        return int(tid)
    except (TypeError, ValueError):
        return None

def list_all_tracks(anns_by_frame):
    """Return sorted [(category_id, track_id, frame_count)] across all frames in a clip."""
    counts = {}
    for anns in anns_by_frame.values():
        for ann in anns:
            tid = get_track_id(ann)
            if tid is None:
                continue
            key = (ann["category_id"], tid)
            counts[key] = counts.get(key, 0) + 1
    return sorted(
        [(cat, tid, n) for (cat, tid), n in counts.items()],
        key=lambda x: (x[0], x[1]),
    )

def load_hoi(hoi_path):
    if hoi_path.exists():
        with open(hoi_path, "r") as f:
            return json.load(f).get("annotations", [])
    return []

def save_hoi(hoi_path, annotations):
    with open(hoi_path, "w") as f:
        json.dump({"annotations": annotations}, f, indent=2)

def draw_frame(image_path, anns, show_boxes=True, show_labels=True, highlight=None):
    """Returns RGB ndarray with bboxes + labels overlaid.

    highlight : (category_id, track_id) tuple or None. The matching ann (if any
                in this frame) gets a thick white halo around its bbox, drawn
                last so it's always visible.
    """
    img = cv2.imread(str(image_path))
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    if not anns or (not show_boxes and not show_labels and highlight is None):
        return img

    palette = {1: (0, 255, 0), 2: (255, 128, 0), 3: (0, 200, 255), 4: (255, 0, 200)}
    font       = cv2.FONT_HERSHEY_SIMPLEX
    font_scale = 0.7
    font_thick = 2

    if show_boxes:
        for ann in anns:
            x, y, w, h = ann["bbox"]
            x1, y1 = int(x), int(y)
            x2, y2 = int(x + w), int(y + h)
            color = palette.get(ann["category_id"], (255, 255, 255))
            cv2.rectangle(img, (x1, y1), (x2, y2), color, 2)

    if show_labels:
        anns_by_area = sorted(anns, key=lambda a: a["bbox"][2] * a["bbox"][3], reverse=True)
        for ann in anns_by_area:
            x, y, w, h = ann["bbox"]
            x1, y1 = int(x), int(y)
            color = palette.get(ann["category_id"], (255, 255, 255))

            cat = OBJECT_CATEGORY_NAMES.get(ann["category_id"], "?")
            tid = get_track_id(ann)
            label = f"{cat} #{tid}" if tid is not None else cat

            (tw, th), baseline = cv2.getTextSize(label, font, font_scale, font_thick)
            pad = 3
            text_x = x1
            text_y = max(y1 - pad, th + pad)

            cv2.rectangle(img, (text_x - pad, text_y - th - pad),
                          (text_x + tw + pad, text_y + baseline + pad), color, -1)
            cv2.putText(img, label, (text_x, text_y), font, font_scale, (0, 0, 0),
                        font_thick, lineType=cv2.LINE_AA)

    # Highlight pass — drawn last so it's always on top
    if highlight is not None:
        h_cat, h_tid = highlight
        for ann in anns:
            if ann["category_id"] == h_cat and get_track_id(ann) == h_tid:
                x, y, w, h = ann["bbox"]
                x1, y1 = int(x), int(y)
                x2, y2 = int(x + w), int(y + h)
                # Outer white halo
                cv2.rectangle(img, (x1 - 4, y1 - 4), (x2 + 4, y2 + 4), (255, 255, 255), 4)
                # Inner colored line so the original color is still visible
                color = palette.get(ann["category_id"], (255, 255, 255))
                cv2.rectangle(img, (x1, y1), (x2, y2), color, 3)
                # Big readable label above the halo
                cat_name = OBJECT_CATEGORY_NAMES.get(ann["category_id"], "?")
                hl_label = f">> {cat_name} #{h_tid}"
                (tw, th), baseline = cv2.getTextSize(hl_label, font, 1.0, 3)
                ty = max(y1 - 12, th + 6)
                cv2.rectangle(img, (x1 - 4, ty - th - 6),
                              (x1 - 4 + tw + 8, ty + baseline + 4), (255, 255, 255), -1)
                cv2.putText(img, hl_label, (x1, ty), font, 1.0, (0, 0, 0), 3,
                            lineType=cv2.LINE_AA)
                break  # only highlight one match
    return img

def crop_zoom(img, zoom, cx_pct, cy_pct):
    """Crop a centered window of size (W/zoom, H/zoom) at (cx_pct, cy_pct) of the image."""
    if zoom <= 1.0:
        return img
    H, W = img.shape[:2]
    win_w = int(W / zoom)
    win_h = int(H / zoom)
    cx = int(W * cx_pct / 100)
    cy = int(H * cy_pct / 100)
    x1 = max(0, min(W - win_w, cx - win_w // 2))
    y1 = max(0, min(H - win_h, cy - win_h // 2))
    return img[y1:y1 + win_h, x1:x1 + win_w]

## Annotator UI

Workflow:
1. Pick viewpoint + clip → frames load.
2. Drag the **frame slider** to scrub. Bboxes + track IDs render live.
3. Drag the **start/end range slider** to mark an HOI interval. (Snaps to existing frame numbers.)
4. Pick **action**, fill **person_track_id** and **object_track_id**, click **Add**.
5. Repeat for concurrent / additional HOIs.
6. Click **Save** to write `hoi-anns.json`.

Track IDs come from the bbox labels visible in the preview. If your `anns.json` doesn't yet have `track_id` (deepsort_annotator hasn't run), `object_track_id` can be left as -1.

In [7]:
# Mutable state for the current clip session
state = {
    "viewpoint": None,
    "clip": None,
    "frames_by_number": {},
    "frame_numbers": [],   # sorted list of available frame numbers
    "anns_by_frame": {},
    "hoi_path": None,
    "annotations": [],
    "next_id": 0,
}

In [ ]:
# All widgets + handlers + display in one cell so the widget you see, the widget
# the handlers reference, and the widget that gets observed are guaranteed to be
# the same Python object. Splitting these into separate cells caused stale-widget
# desync where the rendered dropdown wasn't connected to the Python kernel.

# --- Widgets ---
viewpoint_dd = widgets.Dropdown(options=list_viewpoints(), description="Viewpoint")
clip_dd      = widgets.Dropdown(options=list_clips(viewpoint_dd.value) if viewpoint_dd.value else [],
                                 description="Clip")
load_btn     = widgets.Button(description="Load clip", button_style="primary")

frame_slider = widgets.SelectionSlider(options=[0], description="Frame",
                                        layout=widgets.Layout(width="95%"))

show_boxes_cb  = widgets.Checkbox(value=True, description="Show boxes")
show_labels_cb = widgets.Checkbox(value=True, description="Show labels")
zoom_dd        = widgets.Dropdown(options=[("1x", 1.0), ("1.5x", 1.5), ("2x", 2.0),
                                            ("3x", 3.0), ("4x", 4.0)],
                                   value=1.0, description="Zoom")
center_x_slider = widgets.IntSlider(value=50, min=0, max=100, step=1,
                                     description="Center X%",
                                     layout=widgets.Layout(width="45%"))
center_y_slider = widgets.IntSlider(value=50, min=0, max=100, step=1,
                                     description="Center Y%",
                                     layout=widgets.Layout(width="45%"))

highlight_dd = widgets.Dropdown(
    options=[("(none)", None)],
    value=None,
    description="Highlight",
    layout=widgets.Layout(width="50%"),
)

class_filter_cbs = {
    cat_id: widgets.Checkbox(value=True, description=name,
                              layout=widgets.Layout(width="auto"),
                              indent=False)
    for cat_id, name in OBJECT_CATEGORY_NAMES.items()
}

preview_out = widgets.Output()

range_slider = widgets.SelectionRangeSlider(options=[0], index=(0, 0),
                                             description="Interval",
                                             layout=widgets.Layout(width="95%"))
action_dd    = widgets.Dropdown(options=list(ACTION_NAME_TO_ID.keys()), description="Action")
person_id_in = widgets.IntText(value=1, description="Person ID")
object_id_in = widgets.IntText(value=-1, description="Object ID")
add_btn      = widgets.Button(description="Add annotation", button_style="success")

anns_out     = widgets.Output()
delete_id_in = widgets.IntText(value=0, description="Delete id")
delete_btn   = widgets.Button(description="Delete", button_style="warning")
save_btn     = widgets.Button(description="Save hoi-anns.json", button_style="info")
status_out   = widgets.Output()


# --- Handlers ---
def render_preview():
    if not state["frame_numbers"]:
        return
    fn = frame_slider.value
    img_path = state["frames_by_number"][fn]

    enabled_cats = {cat_id for cat_id, cb in class_filter_cbs.items() if cb.value}
    anns = [a for a in state["anns_by_frame"].get(fn, [])
            if a["category_id"] in enabled_cats]

    img = draw_frame(img_path, anns,
                     show_boxes=show_boxes_cb.value,
                     show_labels=show_labels_cb.value,
                     highlight=highlight_dd.value)
    img = crop_zoom(img, zoom_dd.value, center_x_slider.value, center_y_slider.value)
    with preview_out:
        clear_output(wait=True)
        fig = plt.figure(figsize=(12, 7))
        plt.imshow(img)
        plt.title(f"vp{state['viewpoint']} clip{state['clip']}  |  frame {fn}  |  zoom {zoom_dd.value}x")
        plt.axis("off")
        plt.show()
        plt.close(fig)

def render_table():
    with anns_out:
        clear_output(wait=True)
        if not state["annotations"]:
            print("(no annotations yet)")
            return
        for a in state["annotations"]:
            action_name = ACTION_ID_TO_NAME.get(a["action_id"], "?")
            print(f"id={a['id']:>3}  person#{a['person_track_id']}  "
                  f"obj#{a['object_track_id']}  "
                  f"action={action_name}({a['action_id']})  "
                  f"frames {a['start_frame']}..{a['end_frame']}")

def on_viewpoint_change(change):
    clip_dd.options = list_clips(change["new"])

def on_load(_):
    vp = viewpoint_dd.value
    cl = clip_dd.value
    if vp is None or cl is None:
        with status_out:
            clear_output(wait=True)
            print(f"Cannot load: viewpoint={vp!r}, clip={cl!r}")
        return

    frames_by_number, anns_by_frame, hoi_path = load_clip(vp, cl)
    frame_numbers = sorted(frames_by_number.keys())

    state.update({
        "viewpoint": vp,
        "clip": cl,
        "frames_by_number": frames_by_number,
        "frame_numbers": frame_numbers,
        "anns_by_frame": anns_by_frame,
        "hoi_path": hoi_path,
        "annotations": load_hoi(hoi_path),
    })
    state["next_id"] = (max((a["id"] for a in state["annotations"]), default=-1) + 1)

    frame_slider.options = frame_numbers
    frame_slider.value = frame_numbers[0]
    range_slider.options = frame_numbers
    range_slider.index = (0, len(frame_numbers) - 1)

    track_options = [("(none)", None)]
    for cat_id, tid, n_frames in list_all_tracks(anns_by_frame):
        cat_name = OBJECT_CATEGORY_NAMES.get(cat_id, f"cat{cat_id}")
        track_options.append((f"{cat_name} #{tid}  ({n_frames} frames)", (cat_id, tid)))
    highlight_dd.options = track_options
    highlight_dd.value = None

    with status_out:
        clear_output(wait=True)
        print(f"Loaded vp{vp} clip{cl}: {len(frame_numbers)} frames, "
              f"{len(track_options) - 1} tracks, "
              f"{len(state['annotations'])} existing HOI annotations")
    render_preview()
    render_table()

def on_view_change(_):
    render_preview()

def on_add(_):
    if not state["frame_numbers"]:
        return
    start_f, end_f = range_slider.value
    new_ann = {
        "id": state["next_id"],
        "person_track_id": int(person_id_in.value),
        "object_track_id": int(object_id_in.value),
        "action_id": ACTION_NAME_TO_ID[action_dd.value],
        "start_frame": int(start_f),
        "end_frame": int(end_f),
    }
    state["annotations"].append(new_ann)
    state["next_id"] += 1
    render_table()

def on_delete(_):
    target = int(delete_id_in.value)
    state["annotations"] = [a for a in state["annotations"] if a["id"] != target]
    render_table()

def on_save(_):
    if state["hoi_path"] is None:
        return
    save_hoi(state["hoi_path"], state["annotations"])
    with status_out:
        clear_output(wait=True)
        print(f"Saved {len(state['annotations'])} annotations to {state['hoi_path']}")


# --- Wire up handlers ---
viewpoint_dd.observe(on_viewpoint_change, names="value")
load_btn.on_click(on_load)
add_btn.on_click(on_add)
delete_btn.on_click(on_delete)
save_btn.on_click(on_save)

for w in (frame_slider, show_boxes_cb, show_labels_cb, zoom_dd,
          center_x_slider, center_y_slider, highlight_dd, *class_filter_cbs.values()):
    w.observe(on_view_change, names="value")


# --- Layout + display ---
selector_box   = widgets.HBox([viewpoint_dd, clip_dd, load_btn])
class_filter_box = widgets.HBox(
    [widgets.HTML("<b>Show classes:</b>")] + list(class_filter_cbs.values())
)
view_controls  = widgets.VBox([
    widgets.HBox([show_boxes_cb, show_labels_cb, zoom_dd]),
    widgets.HBox([center_x_slider, center_y_slider]),
    class_filter_box,
    highlight_dd,
])
entry_box      = widgets.VBox([
    range_slider,
    widgets.HBox([action_dd, person_id_in, object_id_in, add_btn]),
])
table_controls = widgets.HBox([delete_id_in, delete_btn, save_btn])

display(selector_box,
        status_out,
        frame_slider,
        view_controls,
        preview_out,
        widgets.HTML("<h4>Add HOI interval</h4>"),
        entry_box,
        widgets.HTML("<h4>Annotations for this clip</h4>"),
        anns_out,
        table_controls)

Output()

SelectionSlider(description='Frame', layout=Layout(width='95%'), options=(0,), value=0)

Output()

HTML(value='<h4>Add HOI interval</h4>')

HTML(value='<h4>Annotations for this clip</h4>')

Output()